# 오디오 LLM 하이햇·라이드 마디 판정 — GPU 서버 실행 (S15P31A307-325)

소리를 듣는 공개 LLM 8개에 한 마디 길이 오디오를 들려주고, 박을 세는 심벌이 하이햇(H)인지 라이드(R)인지 묻는다.

- 모델: Qwen3-Omni, Nemotron-3-Nano-Omni, Voxtral-Small, Audio Flamingo Next, MOSS-Music, MiMo-Audio, Step-Audio-2-mini, Kimi-Audio. 한 번에 하나씩 GPU 6번에 올린다
- 순서: 모델마다 가상환경 만들기 → 가중치 받기 → 작은 시험 120마디(원곡·분리 드럼) → 통과 판단 → 통과한 오디오만 전체 3,513마디
- 결과: `~/ssaplash/results/cymrole_audio.zip` (답 파일·로그·통과 판단). 가상환경·가중치·저장소는 `~/ssaplash/cymrole_audio/` 아래에만 둔다

**준비물.** 로컬 `AI/`에서 `uv run python scripts/cymrole_audio_cases.py --out data/results/cymrole_audio --pack-only --pack data/upload/server/cymrole_audio.zip`로 만든 zip(약 420MB)을 JupyterHub `ssaplash/upload/`에, 이 파일을 `ssaplash/notebooks/`에 올린다. 운영 코드 `~/ssaplash/S15P31A307`과 그 `.venv`는 쓰지도 실행하지도 않는다.

**실행.** 1번 셀을 한 번 실행하면 끝까지 백그라운드로 돈다. 브라우저를 닫아도 계속 돈다. 진행은 2번 셀로 아무 때나 본다.

**끊기면.** 서버가 다시 시작됐거나 2번 셀에 '멈춤 또는 끝'인데 실패 단계가 있으면 1번 셀을 다시 실행한다. 끝난 단계는 건너뛰고, 판정은 이미 답한 마디 다음부터 이어 간다. 실패한 모델만 다시 시도한다. 이미 돌고 있을 때 1번을 다시 치면 새로 시작하지 않는다.


In [ ]:
# 1. 시작 (여러 번 실행해도 됨). 처음이면 zip을 풀고 전체 과정을 백그라운드로 시작함
#    이미 돌고 있으면 새로 시작하지 않고, 멈춰 있으면 끝난 단계는 건너뛰고 이어 감
import os
import subprocess
import sys
import zipfile
from pathlib import Path

GPU_ID = "6"                                   # 지정받은 GPU
MODELS = ""                                    # 빈 칸이면 8개 전부. 일부만: "qwen3omni voxtral"
WORK = Path.home() / "ssaplash"
UPLOAD = WORK / "upload" / "cymrole_audio.zip"
CA = WORK / "cymrole_audio"                    # 이 작업 폴더
PIPE = CA / "scripts" / "cymrole_audio_pipeline.py"

# zip이 새로 올라왔을 때만 풂 (질문·정답·마디 wav·스크립트. 답 파일은 zip에 없어서 안 지워짐)
mark = CA / ".unzipped"
if not mark.exists() or UPLOAD.stat().st_mtime > mark.stat().st_mtime:
    with zipfile.ZipFile(UPLOAD) as z:
        z.extractall(WORK)
    mark.write_text(str(UPLOAD.stat().st_mtime))
    print("zip 풂:", UPLOAD.stat().st_size // 10**6, "MB")

uv = f"{sys.executable} -m uv"
if subprocess.run([sys.executable, "-m", "uv", "--version"], capture_output=True).returncode != 0:
    uv = "uv"                                  # 커널 파이썬에 uv가 없으면 시스템 uv

pid_file = CA / "pipeline.pid"
running = False
if pid_file.exists():
    try:
        os.kill(int(pid_file.read_text()), 0)
        running = True
    except OSError:
        pass
if running:
    print("이미 도는 중. pid", pid_file.read_text())
else:
    env = dict(os.environ)
    env.pop("PYTHONPATH", None)
    cmd = [sys.executable, str(PIPE), "run", "--work", str(CA), "--results", str(WORK / "results"),
           "--gpu", GPU_ID, "--uv", uv]
    if MODELS.strip():
        cmd += ["--models", *MODELS.split()]
    proc = subprocess.Popen(["nohup", "nice", "-n", "10", *cmd], cwd=CA, env=env,
                            stdout=open(CA / "pipeline.log", "a"), stderr=subprocess.STDOUT,
                            start_new_session=True)
    print("시작. pid", proc.pid, "로그", CA / "pipeline.log")
# 확인: 처음엔 'zip 풂: 42x MB'와 '시작. pid 숫자 로그 /home/.../ssaplash/cymrole_audio/pipeline.log'.
#       다시 치면 '이미 도는 중. pid 숫자'. 1분쯤 뒤 2번 셀에서 qwen3omni 줄이 env:- 에서 진행되는지 봄


In [ ]:
# 2. 진행 확인 (여러 번 쳐도 됨. 커널이 다시 시작돼도 이 셀만 바로 쳐도 됨)
import sys
from pathlib import Path

CA = Path.home() / "ssaplash" / "cymrole_audio"
!{sys.executable} {CA}/scripts/cymrole_audio_pipeline.py status --work {CA}
!nvidia-smi -i 6 --query-gpu=memory.used,memory.total,utilization.gpu --format=csv,noheader
!cat {CA}/check.json 2>/dev/null | head -12
!df -h {CA} | tail -1
!ls -la ~/ssaplash/results/cymrole_audio.zip 2>/dev/null || echo "결과 zip 아직 없음"
# 확인 (도는 중 예):
#   파이프라인 도는 중 (pid 12345)
#     qwen3omni   env:끝 download:끝 pilot:- gate:- full:-
#     ...
#     답 qwen3omni_mix: 57마디
#   파이프라인 로그 끝: [11-10 10:00:00] 시작 pilot_qwen3omni
#   지금 단계 pilot_qwen3omni 로그 끝: [llm] qwen3omni_mix: 50/120, 마디당 0.80초
# - 가상환경 하나 만드는 데 5~15분, 30B 모델 가중치 받기에 수십 분이 걸릴 수 있음
# - check.json의 gpu_free_gb가 70보다 작으면 Qwen3-Omni·Nemotron이 메모리 부족으로 실패할 수 있음. 실패 줄을 알려 줌
# - 끝: '멈춤 또는 끝'과 로그 끝에 '전체 끝', zip 크기가 보이면 zip을 내려받음. 실패 단계가 있으면 그 줄을 알려 줌
